# TourAPI 전국 관광지 마스터 수집

- **목적**: 클러스터링용 관광지 위경도·카테고리·행정구역 확보
- **수집 대상**: 관광지(12), 문화시설(14), 레포츠(28)
- **출력**: `output/tour_master.csv`

> ⚠️ 주변시설(음식점·숙박·쇼핑)은 나중에 분산 후보 추린 뒤 따로 수집

## 0. 설정 — API 키 입력

In [1]:
import requests
import pandas as pd
import time
from pathlib import Path

# ─── 여기에 발급받은 키 입력 ───────────────────────────
# 공공데이터포털 마이페이지의 '일반 인증키(Decoding)' 사용
SERVICE_KEY = "4b661689a95022eb385895ed069b9f18924b54efb86501eed35e42be243d08bf"
# ────────────────────────────────────────────────────────

OUTPUT_DIR = Path("./output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONTENT_TYPES = {12: "관광지", 14: "문화시설", 28: "레포츠"}
TARGET_TYPES = [12, 14, 28]

## 1. API 버전 확인

TourAPI는 `KorService1`(구버전)과 `KorService2`(신버전)이 공존합니다.
발급 시점에 따라 다르므로, 어느 쪽이 동작하는지 먼저 확인합니다.

In [2]:
def test_endpoint(base_url):
    """엔드포인트 동작 테스트"""
    url = f"{base_url}/areaBasedList1" if 'KorService1' in base_url else f"{base_url}/areaBasedList2"
    params = {
        "serviceKey": SERVICE_KEY,
        "numOfRows": 1, "pageNo": 1,
        "MobileOS": "ETC", "MobileApp": "TourRecommender",
        "_type": "json", "contentTypeId": 12,
    }
    try:
        r = requests.get(url, params=params, timeout=15)
        if r.status_code == 200 and 'response' in r.text:
            data = r.json()
            header = data['response']['header']
            code = header.get('resultCode', '?')
            msg = header.get('resultMsg', '?')
            if code == '0000':
                total = data['response']['body']['totalCount']
                print(f"  ✅ 성공! 전체 관광지 수: {total:,}건")
                return True
            else:
                print(f"  ⚠️ 응답 코드: {code} / {msg}")
                return False
        else:
            print(f"  ❌ HTTP {r.status_code}")
            print(f"  응답 일부: {r.text[:200]}")
            return False
    except Exception as e:
        print(f"  ❌ 오류: {e}")
        return False

endpoints = {
    "KorService2 (신버전)": "http://apis.data.go.kr/B551011/KorService2",
    "KorService1 (구버전)": "http://apis.data.go.kr/B551011/KorService1",
}

BASE_URL = None
for name, url in endpoints.items():
    print(f"테스트: {name}")
    if test_endpoint(url):
        BASE_URL = url
        print(f"\n👉 사용할 엔드포인트: {name}\n   {url}")
        break
    print()

if BASE_URL is None:
    print("\n⚠️ 둘 다 실패. 아래를 확인하세요:")
    print("  1. API 키가 정확한지 (Decoding 키 사용)")
    print("  2. 활용신청이 승인됐는지 (마이페이지)")
    print("  3. 승인 후 1시간 정도 지났는지")

테스트: KorService2 (신버전)
  ✅ 성공! 전체 관광지 수: 12,715건

👉 사용할 엔드포인트: KorService2 (신버전)
   http://apis.data.go.kr/B551011/KorService2


## 2. 수집 함수

In [7]:
# 셀 2 전체 교체 — KorService2 맞게 수정

SUFFIX = "2" if "KorService2" in BASE_URL else "1"

def get_area_list(content_type_id, page_no=1, num_of_rows=1000):
    """지역기반 관광정보 목록 조회 (KorService2 호환)"""
    url = f"{BASE_URL}/areaBasedList{SUFFIX}"
    params = {
        "serviceKey": SERVICE_KEY,
        "numOfRows": num_of_rows, "pageNo": page_no,
        "MobileOS": "ETC", "MobileApp": "TourRecommender",
        "_type": "json", "arrange": "A",   # listYN 제거
        "contentTypeId": content_type_id,
    }
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    # KorService2는 response 래퍼 없이 바로 응답
    if 'response' in data:
        return data['response']     # 구버전
    return data                     # 신버전

def collect_type(content_type_id):
    """특정 콘텐츠 타입 전체 수집"""
    name = CONTENT_TYPES[content_type_id]
    print(f"\n[{name}] 수집 시작...")

    first = get_area_list(content_type_id, page_no=1)
    total = int(first['body']['totalCount'])
    pages = (total + 999) // 1000
    print(f"  총 {total:,}건 / {pages}페이지")

    items = []
    for p in range(1, pages + 1):
        try:
            data = get_area_list(content_type_id, page_no=p)
            body = data['body']['items']
            if not body:
                break
            lst = body.get('item', [])
            if isinstance(lst, dict):
                lst = [lst]
            items.extend(lst)
            print(f"  페이지 {p}/{pages} ({len(lst)}건)", end='\r')
            time.sleep(0.2)
        except Exception as e:
            print(f"\n  페이지 {p} 오류: {e}")
            time.sleep(1)

    print(f"\n  [{name}] 완료: {len(items)}건")
    return items

## 3. 전체 수집 실행

In [8]:
all_items = []
for ct_id in TARGET_TYPES:
    items = collect_type(ct_id)
    for it in items:
        it['content_type_name'] = CONTENT_TYPES[ct_id]
    all_items.extend(items)

print(f"\n전체 수집 완료: {len(all_items):,}건")


[관광지] 수집 시작...
  총 12,715건 / 13페이지
  페이지 13/13 (715건))
  [관광지] 완료: 12715건

[문화시설] 수집 시작...
  총 2,658건 / 3페이지
  페이지 3/3 (658건))
  [문화시설] 완료: 2658건

[레포츠] 수집 시작...
  총 3,893건 / 4페이지
  페이지 4/4 (893건))
  [레포츠] 완료: 3893건

전체 수집 완료: 19,266건


## 4. 정리 및 저장

In [9]:
df = pd.DataFrame(all_items)

# 핵심 컬럼만 추출 + 이름 정리
keep = {
    'contentid': 'content_id',
    'contenttypeid': 'content_type_id',
    'content_type_name': 'content_type_name',
    'title': 'name',
    'addr1': 'address',
    'areacode': 'area_code',
    'sigungucode': 'sigungu_code',
    'cat1': 'cat1', 'cat2': 'cat2', 'cat3': 'cat3',
    'mapx': 'longitude', 'mapy': 'latitude',
    'tel': 'tel', 'firstimage': 'image_url',
}
avail = {k: v for k, v in keep.items() if k in df.columns}
df = df[list(avail.keys())].rename(columns=avail)

# 위경도 숫자화 + 결측 제거 (클러스터링 필수)
df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
before = len(df)
df = df.dropna(subset=['latitude', 'longitude'])
df = df[(df['latitude'] > 33) & (df['latitude'] < 39) &
        (df['longitude'] > 124) & (df['longitude'] < 132)]  # 한국 영역
print(f"위경도 정제: {before:,} → {len(df):,}건")

# 중복 제거
df = df.drop_duplicates(subset='content_id')

# 저장
out = OUTPUT_DIR / 'tour_master.csv'
df.to_csv(out, index=False, encoding='utf-8-sig')
print(f"\n✅ 저장: {out}")
print(f"   관광지 수: {len(df):,}건")
print(f"   타입 분포:\n{df['content_type_name'].value_counts().to_string()}")
df.head()

위경도 정제: 19,266 → 19,223건

✅ 저장: output\tour_master.csv
   관광지 수: 19,223건
   타입 분포:
content_type_name
관광지     12680
레포츠      3890
문화시설     2653


,content_id,content_type_id,content_type_name,name,address,area_code,sigungu_code,cat1,cat2,cat3,longitude,latitude,tel,image_url
0,127480,12,관광지,가거도,전라남도 신안군 흑산면 가거도길 38-2,38,12,A01,A0101,A01011300,125.126386,34.052061,,http://tong.visitkorea.or.kr/cms/resource/28/3...
1,126273,12,관광지,가계해수욕장,전라남도 진도군 고군면 신비의바닷길 47,38,21,A01,A0101,A01011200,126.354741,34.435459,,http://tong.visitkorea.or.kr/cms/resource/36/3...
2,2019720,12,관광지,가고파 꼬부랑길 벽화마을,경상남도 창원시 마산합포구 성호서7길 15-8,,,,,,128.569630,35.207726,,https://tong.visitkorea.or.kr/cms/resource/55/...
3,2994116,12,관광지,가곡유황온천&스파,강원특별자치도 삼척시 가곡면 가곡천로 1510,32,4,A02,A0202,A02020300,129.206449,37.150902,,http://tong.visitkorea.or.kr/cms/resource/54/2...
4,3534460,12,관광지,가곡유황족욕체험장,강원특별자치도 삼척시 가곡면 가곡천로 1476,,,,,,129.205955,37.150069,,https://tong.visitkorea.or.kr/cms/resource/73/...


In [10]:
# 시도별 분포 확인 (area_code 기준)
AREA_NAMES = {
    '1': '서울', '2': '인천', '3': '대전', '4': '대구', '5': '광주',
    '6': '부산', '7': '울산', '8': '세종', '31': '경기', '32': '강원',
    '33': '충북', '34': '충남', '35': '경북', '36': '경남',
    '37': '전북', '38': '전남', '39': '제주',
}
df['area_name'] = df['area_code'].astype(str).map(AREA_NAMES)
print("시도별 관광지 수:")
print(df['area_name'].value_counts().to_string())

시도별 관광지 수:
area_name
경기    2028
강원    1533
경북    1285
경남    1238
전남    1085
충남     991
전북     963
충북     866
서울     818
제주     560
인천     359
대구     318
부산     261
울산     166
광주     149
대전     130
세종      40


In [11]:
import pandas as pd
df = pd.read_csv('output/tour_master.csv')
print(df[['name','cat1','cat2','cat3']].head(10).to_string())
print(f"\ncat1 유니크: {df['cat1'].nunique()}개")
print(df['cat1'].value_counts().to_string())

            name cat1   cat2       cat3
0            가거도  A01  A0101  A01011300
1         가계해수욕장  A01  A0101  A01011200
2  가고파 꼬부랑길 벽화마을  NaN    NaN        NaN
3      가곡유황온천&스파  A02  A0202  A02020300
4      가곡유황족욕체험장  NaN    NaN        NaN
5      가금철교 문화공원  A02  A0202  A02020700
6         가나아트파크  A02  A0202  A02020600
7         가남체육공원  A02  A0202  A02020700
8            가덕도  NaN    NaN        NaN
9         가덕도 등대  A01  A0101  A01011600

cat1 유니크: 3개
cat1
A02    7371
A03    3093
A01    2330


In [12]:
# cat1~cat3 코드 의미 확인 + NaN 비율
print("=== cat1 분포 ===")
cat1_map = {'A01': '자연', 'A02': '인문(문화·역사·체험)', 'A03': '레포츠'}
print(df['cat1'].map(cat1_map).value_counts())
print(f"NaN: {df['cat1'].isna().sum()}개 ({df['cat1'].isna().mean()*100:.1f}%)")

print("\n=== cat2 분포 (상위 15) ===")
print(df['cat2'].value_counts().head(15).to_string())

print("\n=== NaN인 관광지 샘플 ===")
print(df[df['cat1'].isna()][['name','content_type_name','address']].head(10).to_string())

=== cat1 분포 ===
cat1
인문(문화·역사·체험)    7371
레포츠             3093
자연              2330
Name: count, dtype: int64
NaN: 6429개 (33.4%)

=== cat2 분포 (상위 15) ===
cat2
A0201    2741
A0302    2724
A0101    2203
A0206    1488
A0203    1349
A0202    1184
A0205     438
A0303     305
A0204     171
A0102     127
A0301      24
A0305      22
A0304      18

=== NaN인 관광지 샘플 ===
             name content_type_name                      address
2   가고파 꼬부랑길 벽화마을               관광지    경상남도 창원시 마산합포구 성호서7길 15-8
4       가곡유황족욕체험장               관광지    강원특별자치도 삼척시 가곡면 가곡천로 1476
8             가덕도               관광지  부산광역시 강서구 서천로42번길 351 (천성동)
11      가덕도외양포전망대               관광지      부산광역시 강서구 외양포로 10 (대항동)
13            가라산               관광지             경상남도 거제시 남부면 다대리
17           가령폭포               관광지          강원특별자치도 홍천군 내촌면 와야리
20        가리산(홍천)               관광지   강원특별자치도 홍천군 두촌면 가리산길 260-9
21       가리산자연휴양림               관광지     강원특별자치도 홍천군 두촌면 가리산길 426
27          가마소계곡               관광지          전북특별자치도 

In [13]:
# cat2 코드 → 의미 매핑 확인
cat2_map = {
    'A0101': '자연관광지', 'A0102': '관광자원',
    'A0201': '역사관광지', 'A0202': '휴양관광지',
    'A0203': '체험관광지', 'A0204': '산업관광지',
    'A0205': '건축/조형물', 'A0206': '문화시설',
    'A0301': '레포츠소개', 'A0302': '육상레포츠',
    'A0303': '수상레포츠', 'A0304': '항공레포츠',
    'A0305': '복합레포츠',
}
df['cat2_name'] = df['cat2'].map(cat2_map)
print("=== cat2 분포 (이름) ===")
print(df['cat2_name'].value_counts().to_string())

# NaN → content_type_name으로 cat1 추정
print("\n=== NaN 관광지의 content_type_name 분포 ===")
print(df[df['cat1'].isna()]['content_type_name'].value_counts())

# NaN 채우기 시도 — content_type으로 추정
type_to_cat1 = {'관광지': 'A01', '문화시설': 'A02', '레포츠': 'A03'}
df['cat1_filled'] = df['cat1'].fillna(df['content_type_name'].map(type_to_cat1))
df['cat1_filled_name'] = df['cat1_filled'].map({'A01':'자연','A02':'인문','A03':'레포츠'})
print(f"\ncat1 채운 후 NaN: {df['cat1_filled'].isna().sum()}개")
print(df['cat1_filled_name'].value_counts().to_string())

=== cat2 분포 (이름) ===
cat2_name
역사관광지     2741
육상레포츠     2724
자연관광지     2203
문화시설      1488
체험관광지     1349
휴양관광지     1184
건축/조형물     438
수상레포츠      305
산업관광지      171
관광자원       127
레포츠소개       24
복합레포츠       22
항공레포츠       18

=== NaN 관광지의 content_type_name 분포 ===
content_type_name
관광지     4468
문화시설    1164
레포츠      797
Name: count, dtype: int64

cat1 채운 후 NaN: 0개
cat1_filled_name
인문     8535
자연     6798
레포츠    3890
